In [ ]:
# Exercise 4. (When was Tom in the class?)
# Consider the example ”When was Tom in the class?” from the course ”AI and ML”. The
# knowledge base KB is given by the logical term:
# KB=(M∨T)∧[(M ∧¬W)∨(¬M ∧W)]∧(T ∨W)
# where M,T,W are propositional variables. Verfify that KB |= T, i.e. KB entails T by
#    a) considering the truth tables of KB and T,
#    b) comparing the models of KB and T,
#    c) applying the function satisfiable() to the term KB ∨ ¬T

In [3]:
# a)
from sympy import symbols
from sympy.logic.boolalg import truth_table

M, T, W = symbols('M T W')

KB = (M | T) & ((M & ~W) | (~M & W)) & (T | W)

print("M     T     W  |  KB   |  T")
print("-" * 35)
for (m, t, w), kb_val in truth_table(KB, [M, T, W]):
    print(f"{int(bool(m))}     {int(bool(t))}     {int(bool(w))}  |   {int(bool(kb_val))}   |  {int(bool(t))}")

M     T     W  |  KB   |  T
-----------------------------------
0     0     0  |   0   |  0
0     0     1  |   0   |  0
0     1     0  |   0   |  1
0     1     1  |   1   |  1
1     0     0  |   0   |  0
1     0     1  |   0   |  0
1     1     0  |   1   |  1
1     1     1  |   0   |  1


In [4]:
#b)
from sympy.logic.inference import satisfiable

# Collect all models of KB
models_KB = list(satisfiable(KB, all_models=True))
print("Models of KB:")
for m in models_KB:
    print(f"  {m}")

# Collect all models of T (trivially: any assignment where T=True)
models_T = list(satisfiable(T, all_models=True))
print("\nModels of T:")
for m in models_T:
    print(f"  {m}")

# Check subset relationship
# Every model of KB must have T=True
entailed = all(m.get(T, False) for m in models_KB)
print(f"\nKB |= T : {entailed}")

Models of KB:
  {M: True, W: False, T: True}
  {M: False, W: True, T: True}

Models of T:
  {T: True}

KB |= T : True


In [5]:
# c)
# If KB ∧ ¬T is unsatisfiable → KB |= T
result = satisfiable(KB & ~T)

if result == False:
    print("KB ∧ ¬T is UNSATISFIABLE  →  KB |= T  ✓")
else:
    print(f"KB ∧ ¬T is SATISFIABLE  →  KB does NOT entail T")
    print(f"Counterexample: {result}")
    

KB ∧ ¬T is UNSATISFIABLE  →  KB |= T  ✓


In [6]:
# Exercise 5.
#Write two versions of a Python function entails(KB,φ), which returns True if KB |= φ,
#otherwise False by
#    a) comparing the models of KB and φ,
#    b) applying the function satisfiable() to the term KB ∨ ¬φ

In [7]:
# a)
from sympy import symbols
from sympy.logic.boolalg import And
from sympy.logic.inference import satisfiable
from sympy.logic.boolalg import truth_table

def entails_models(KB, phi):
    """KB |= phi iff every model of KB is also a model of phi."""
    
    models_KB = list(satisfiable(KB, all_models=True))
    
    # Edge case: KB is a tautology → {False: False} sentinel
    if models_KB == [{False: False}]:
        # Every valuation satisfies KB, so check if phi is also a tautology
        models_phi = list(satisfiable(phi, all_models=True))
        return models_phi == [{False: False}]
    
    # For each model of KB, check it also satisfies phi
    for model in models_KB:
        # Substitute the model's values into phi
        phi_val = phi.subs(list(model.items()))
        if not bool(phi_val):
            return False  # Found a counterexample
    
    return True


# ── Test with Exercise 4 ─────────────────────────────────────────────
M, T, W = symbols('M T W')
KB = (M | T) & ((M & ~W) | (~M & W)) & (T | W)

print(f"KB |= T  : {entails_models(KB, T)}")   # Expected: True
print(f"KB |= M  : {entails_models(KB, M)}")   # Expected: False
print(f"KB |= W  : {entails_models(KB, W)}")   # Expected: False

KB |= T  : True
KB |= M  : False
KB |= W  : False


In [8]:
#b)
def entails_sat(KB, phi):
    """KB |= phi iff KB ∧ ¬phi is unsatisfiable."""
    
    result = satisfiable(KB & ~phi)
    return result == False  # Unsatisfiable → entailment holds


# ── Test with Exercise 4 ─────────────────────────────────────────────
print(f"KB |= T  : {entails_sat(KB, T)}")   # Expected: True
print(f"KB |= M  : {entails_sat(KB, M)}")   # Expected: False
print(f"KB |= W  : {entails_sat(KB, W)}")   # Expected: False

KB |= T  : True
KB |= M  : False
KB |= W  : False


In [9]:
# cross checking both 
print("Comparing both implementations:")
print(f"  entails_models(KB, T) = {entails_models(KB, T)}")
print(f"  entails_sat(KB, T)    = {entails_sat(KB, T)}")

print(f"  entails_models(KB, M) = {entails_models(KB, M)}")
print(f"  entails_sat(KB, M)    = {entails_sat(KB, M)}")

Comparing both implementations:
  entails_models(KB, T) = True
  entails_sat(KB, T)    = True
  entails_models(KB, M) = False
  entails_sat(KB, M)    = False
